Dans cette partie, nous allons faire le traitement de nos données de a à z

In [1]:
%run "./01_configuration.ipynb"
import json
import numpy as np
import pandas as pd

df_raw = pd.read_csv(DATA_FILE)
df = df_raw.copy()

df.columns = df.columns.str.strip()

print("Dimensions :", df.shape)
display(df.head())

ROOT_DIR  : C:\Users\Admin\Desktop\Projet_Artemis2
DATA_FILE : C:\Users\Admin\Desktop\Projet_Artemis2\data\Artemis.csv | existe: True
RANDOM_SEED: 42 | DEVICE: cuda
CONFIGURATION DU PROJET HESS

BATTERIE ÉNERGIE
Énergie          : 13709.89 Wh
Tension          : 450.00 V
Capacité         : 30.4664 Ah
Masse            : 55.12 kg
Courant recharge : -14.00 A
Courant décharge : 28.00 A
Configuration    : 125S7P

BATTERIE PUISSANCE
Énergie          : 2987.12 Wh
Tension          : 402.60 V
Capacité         : 7.4196 Ah
Masse            : 50.02 kg
Courant recharge : -130.00 A
Courant décharge : 400.00 A
Configuration    : 122S2P

HESS
Énergie totale   : 16697.01 Wh
Masse totale     : 105.14 kg
Tension du bus   : 402.60 V
Puissance min    : -58638.00 W
Puissance max    : 173640.00 W

MODÈLES
LSTM seul        : 7 → 64 → 3
LSTM NS          : 15 → 64 → 3
GNN simple       : 12 → 32 → 1
MLP simple       : 5 → 64 → 32 → 1
MLP NS           : 17 → 64 → 32 → 1

Device           : cuda
Fichier          : 

,time,speed,hasAeroForce,hasRollingForce,hasGravityForce,hasAcceleration,hasAccelerationForce,hasTotalForce,hasPower,P_EB,P_PB,P_conv,I_EB,I_PB,I_load,I_prime,SOC_EB,SOC_PB,SOC_HESS
0,1,0.0,0.0,109.87,0.0,0.0,0.0,109.87,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,2,0.0,0.0,109.87,0.0,0.0,0.0,109.87,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,3,0.0,0.0,109.87,0.0,0.0,0.0,109.87,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
3,4,0.0,0.0,109.87,0.0,0.0,0.0,109.87,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,5,0.0,0.0,109.87,0.0,0.0,0.0,109.87,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0


Vérification des colonnes

In [2]:
real_cols = set(df.columns)
expected_cols = set(DATA_COLUMNS)

missing_cols = sorted(expected_cols - real_cols)
extra_cols = sorted(real_cols - expected_cols)

print("Colonnes manquantes :", missing_cols)
print("Colonnes supplémentaires :", extra_cols)

if missing_cols:
    raise ValueError(
        f"Colonnes obligatoires absentes : {missing_cols}"
    )

Colonnes manquantes : []
Colonnes supplémentaires : []


Conversion numériques

In [3]:
before_na = df.isna().sum()

for col in DATA_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

after_na = df.isna().sum()
new_na = after_na - before_na

conversion_errors = new_na[new_na > 0]

print("Valeurs manquantes :", int(df.isna().sum().sum()))
print("Erreurs de conversion :")
display(conversion_errors)

Valeurs manquantes : 0
Erreurs de conversion :


Series([], dtype: int64)

Valeurs infinies et doublons

In [4]:
numeric_values = df[DATA_COLUMNS].to_numpy(dtype=float)

inf_count = int(np.isinf(numeric_values).sum())
duplicate_rows = int(df.duplicated().sum())
duplicate_time = int(df[TIME_COL].duplicated().sum())

print("Valeurs infinies :", inf_count)
print("Lignes dupliquées :", duplicate_rows)
print("Temps dupliqués :", duplicate_time)

if inf_count > 0:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

if df.isna().any().any():
    problematic = df[df.isna().any(axis=1)]
    display(problematic.head())
    raise ValueError("Des valeurs manquantes ou infinies subsistent.")

Valeurs infinies : 0
Lignes dupliquées : 0
Temps dupliqués : 0


Ordre temporel

In [5]:
df = df.sort_values(TIME_COL).reset_index(drop=True)

is_monotonic = df[TIME_COL].is_monotonic_increasing

print("Temps croissant :", is_monotonic)

if df[TIME_COL].duplicated().any():
    raise ValueError("La colonne time contient des doublons.")

Temps croissant : True


Pas de temps

In [6]:
EPS_POWER_W = 100.0

TIME_TOLERANCE = 1e-9
POWER_TOLERANCE_W = 1e-6
SOC_TOLERANCE = 5e-4

In [7]:
dt = df[TIME_COL].diff().dropna()

if (dt <= 0).any():
    raise ValueError("Le pas de temps contient des valeurs nulles ou négatives.")

DT_SECONDS = float(dt.median())

dt_regular = np.allclose(
    dt.to_numpy(),
    DT_SECONDS,
    atol=TIME_TOLERANCE,
    rtol=0.0,
)

print("DT médian :", DT_SECONDS)
print("DT minimum :", dt.min())
print("DT maximum :", dt.max())
print("Pas de temps régulier :", dt_regular)

if not dt_regular:
    print(dt.value_counts().head(10))

DT médian : 1.0
DT minimum : 1.0
DT maximum : 1.0
Pas de temps régulier : True


dt.mode()[0]

Vérification de l'équilibre de puissance

In [8]:
df["power_balance_error"] = (
    df["hasPower"] - df["P_EB"] - df["P_PB"]
)

power_error = df["power_balance_error"]

print("Erreur moyenne :", power_error.mean())
print("MAE puissance  :", power_error.abs().mean())
print("RMSE puissance :", np.sqrt(np.mean(power_error**2)))
print("Erreur maximale:", power_error.abs().max())

Erreur moyenne : 6.086558954327658e-15
MAE puissance  : 1.2407821959603739e-13
RMSE puissance : 4.5144916748033096e-13
Erreur maximale: 3.637978807091713e-12


In [9]:
power_balance_valid = np.allclose(
    df["hasPower"],
    df["P_EB"] + df["P_PB"],
    atol=POWER_TOLERANCE_W,
    rtol=0.0,
)

print("Équilibre de puissance valide :", power_balance_valid)

Équilibre de puissance valide : True


Vérification des SOC

In [10]:
soc_eb_strict = (
    (df["SOC_EB"] < SOC_EB_MIN)
    | (df["SOC_EB"] > SOC_EB_MAX)
)

soc_pb_strict = (
    (df["SOC_PB"] < SOC_PB_MIN)
    | (df["SOC_PB"] > SOC_PB_MAX)
)

soc_eb_tolerance = (
    (df["SOC_EB"] < SOC_EB_MIN - SOC_TOLERANCE)
    | (df["SOC_EB"] > SOC_EB_MAX + SOC_TOLERANCE)
)

soc_pb_tolerance = (
    (df["SOC_PB"] < SOC_PB_MIN - SOC_TOLERANCE)
    | (df["SOC_PB"] > SOC_PB_MAX + SOC_TOLERANCE)
)

print("SOC_EB violations strictes :", int(soc_eb_strict.sum()))
print("SOC_EB violations réelles  :", int(soc_eb_tolerance.sum()))

print("SOC_PB violations strictes :", int(soc_pb_strict.sum()))
print("SOC_PB violations réelles  :", int(soc_pb_tolerance.sum()))

print("SOC_EB min/max :", df["SOC_EB"].min(), df["SOC_EB"].max())
print("SOC_PB min/max :", df["SOC_PB"].min(), df["SOC_PB"].max())

SOC_EB violations strictes : 439
SOC_EB violations réelles  : 0
SOC_PB violations strictes : 0
SOC_PB violations réelles  : 0
SOC_EB min/max : 0.199749 1.0
SOC_PB min/max : 0.202598 1.0


In [11]:
df["SOC_EB_raw"] = df["SOC_EB"]
df["SOC_PB_raw"] = df["SOC_PB"]

eb_near_lower_bound = (
    (df["SOC_EB"] < SOC_EB_MIN)
    & (df["SOC_EB"] >= SOC_EB_MIN - SOC_TOLERANCE)
)

pb_near_lower_bound = (
    (df["SOC_PB"] < SOC_PB_MIN)
    & (df["SOC_PB"] >= SOC_PB_MIN - SOC_TOLERANCE)
)

df.loc[eb_near_lower_bound, "SOC_EB"] = SOC_EB_MIN
df.loc[pb_near_lower_bound, "SOC_PB"] = SOC_PB_MIN

print("SOC_EB ajustés :", int(eb_near_lower_bound.sum()))
print("SOC_PB ajustés :", int(pb_near_lower_bound.sum()))

SOC_EB ajustés : 439
SOC_PB ajustés : 0


Vérification des puissantes et courants

In [12]:
checks = {
    "P_EB": (
        (df["P_EB"] < P_EB_MIN_W)
        | (df["P_EB"] > P_EB_MAX_W)
    ),
    "P_PB": (
        (df["P_PB"] < P_PB_MIN_W)
        | (df["P_PB"] > P_PB_MAX_W)
    ),
    "I_EB": (
        (df["I_EB"] < I_EB_MIN_A)
        | (df["I_EB"] > I_EB_MAX_A)
    ),
    "I_PB": (
        (df["I_PB"] < I_PB_MIN_A)
        | (df["I_PB"] > I_PB_MAX_A)
    ),
    "P_conv": (
        (df["P_conv"] < P_CONV_MIN_W)
        | (df["P_conv"] > P_CONV_MAX_W)
    ),
}

for name, mask in checks.items():
    print(f"Violations {name} : {int(mask.sum())}")

Violations P_EB : 0
Violations P_PB : 0
Violations I_EB : 0
Violations I_PB : 0
Violations P_conv : 0


Vérification des relations puissances courants

In [13]:
df["P_EB_from_current"] = V_EB_PACK_NOM * df["I_EB"]
df["P_PB_from_current"] = V_PB_PACK_NOM * df["I_PB"]

df["error_P_EB_current"] = (
    df["P_EB"] - df["P_EB_from_current"]
)

df["error_P_PB_current"] = (
    df["P_PB"] - df["P_PB_from_current"]
)

print(
    "MAE P_EB vs V×I :",
    df["error_P_EB_current"].abs().mean()
)

print(
    "MAE P_PB vs V×I :",
    df["error_P_PB_current"].abs().mean()
)

MAE P_EB vs V×I : 0.006657604955272979
MAE P_PB vs V×I : 0.0027196930488800094


Vérification du SOC_HESS

In [14]:
df["SOC_HESS_calculated"] = (
    ENERGY_EB_WH * df["SOC_EB"]
    + ENERGY_PB_WH * df["SOC_PB"]
) / ENERGY_HESS_WH

df["SOC_HESS_error"] = (
    df["SOC_HESS"] - df["SOC_HESS_calculated"]
)

print(
    "MAE SOC_HESS :",
    df["SOC_HESS_error"].abs().mean()
)

print(
    "Erreur max SOC_HESS :",
    df["SOC_HESS_error"].abs().max()
)

MAE SOC_HESS : 3.0371527232069272e-06
Erreur max SOC_HESS : 0.00020621724428507293


Statistiques descriptives

In [15]:
summary_cols = [
    "speed",
    "hasAcceleration",
    "hasPower",
    "P_EB",
    "P_PB",
    "P_conv",
    "I_EB",
    "I_PB",
    "SOC_EB",
    "SOC_PB",
    "SOC_HESS",
]

display(df[summary_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
speed,14530.0,10.660828,8.094549,0.000000,3.055556,10.638889,17.076389,30.972222
hasAcceleration,14530.0,0.000082,0.704734,-4.080000,-0.220000,0.000000,0.280000,2.860000
hasPower,14530.0,3264.011708,10580.243967,-46000.000000,-751.550000,1053.660000,8279.350000,44230.330000
P_EB,14530.0,2715.173752,6027.407289,-6300.000000,-751.550000,466.140000,7622.510000,12600.000000
P_PB,14530.0,548.837956,6253.930256,-39700.000000,0.000000,0.000000,0.000000,43262.710000
P_conv,14530.0,285.998302,634.886901,-663.600000,-79.163250,49.100100,802.904400,1327.200000
I_EB,14530.0,6.033720,13.394238,-14.000000,-1.670100,1.035900,16.938900,28.000000
I_PB,14530.0,1.363234,15.533856,-98.609000,0.000000,0.000000,0.000000,107.458300
SOC_EB,14530.0,0.587983,0.247510,0.200000,0.364989,0.601149,0.808207,1.000000
SOC_PB,14530.0,0.770584,0.148476,0.202598,0.665434,0.778778,0.888395,1.000000


Séparation chronologique

In [16]:
n = len(df)

n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_end = n_train
val_end = n_train + n_val

train_df = df.iloc[:train_end].copy().reset_index(drop=True)
val_df = df.iloc[train_end:val_end].copy().reset_index(drop=True)
test_df = df.iloc[val_end:].copy().reset_index(drop=True)

print("Total :", n)
print("Train :", len(train_df))
print("Validation :", len(val_df))
print("Test :", len(test_df))

print("\nPériodes")
print("Train :", train_df[TIME_COL].iloc[0], "→", train_df[TIME_COL].iloc[-1])
print("Val   :", val_df[TIME_COL].iloc[0], "→", val_df[TIME_COL].iloc[-1])
print("Test  :", test_df[TIME_COL].iloc[0], "→", test_df[TIME_COL].iloc[-1])

Total : 14530
Train : 7265
Validation : 3632
Test : 3633

Périodes
Train : 1 → 7265
Val   : 7266 → 10897
Test  : 10898 → 14530


In [17]:
clean_file = PROCESSED_DIR / "artemis_clean.csv"
train_file = PROCESSED_DIR / "train.csv"
val_file = PROCESSED_DIR / "validation.csv"
test_file = PROCESSED_DIR / "test.csv"

df.to_csv(clean_file, index=False)
train_df.to_csv(train_file, index=False)
val_df.to_csv(val_file, index=False)
test_df.to_csv(test_file, index=False)

print("Fichier complet :", clean_file)
print("Train :", train_file)
print("Validation :", val_file)
print("Test :", test_file)

Fichier complet : C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\artemis_clean.csv
Train : C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\train.csv
Validation : C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\validation.csv
Test : C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\test.csv


In [18]:
quality_report = {
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_rows": duplicate_rows,
    "duplicate_time": duplicate_time,
    "dt_seconds": DT_SECONDS,
    "regular_time_step": bool(dt_regular),
    "power_balance_mae_w": float(power_error.abs().mean()),
    "power_balance_max_w": float(power_error.abs().max()),
    "soc_eb_strict_violations": int(soc_eb_strict.sum()),
    "soc_eb_real_violations": int(soc_eb_tolerance.sum()),
    "soc_pb_strict_violations": int(soc_pb_strict.sum()),
    "soc_pb_real_violations": int(soc_pb_tolerance.sum()),
    "soc_eb_adjusted": int(eb_near_lower_bound.sum()),
    "soc_pb_adjusted": int(pb_near_lower_bound.sum()),
    "p_eb_violations": int(checks["P_EB"].sum()),
    "p_pb_violations": int(checks["P_PB"].sum()),
    "i_eb_violations": int(checks["I_EB"].sum()),
    "i_pb_violations": int(checks["I_PB"].sum()),
    "p_conv_violations": int(checks["P_conv"].sum()),
}

report_file = TABLES_DIR / "data_quality_report.json"

with open(report_file, "w", encoding="utf-8") as file:
    json.dump(quality_report, file, indent=4, ensure_ascii=False)

display(pd.DataFrame(
    quality_report.items(),
    columns=["Indicateur", "Valeur"]
))

print("Rapport enregistré :", report_file)

,Indicateur,Valeur
0,rows,14530
1,columns,28
2,missing_values,0
3,duplicate_rows,0
4,duplicate_time,0
5,dt_seconds,1.0
6,regular_time_step,True
7,power_balance_mae_w,0.0
8,power_balance_max_w,0.0
9,soc_eb_strict_violations,439


Rapport enregistré : C:\Users\Admin\Desktop\Projet_Artemis2\results\tables\data_quality_report.json
